# 「마음돌봄 Vision」 개발 환경 설치 가이드

> **WSL2 (Ubuntu 24.04) 네이티브 설치 기준 · Docker 미사용**
> 작성일: 2026-04-20
> 대상: NVIDIA AI Academy 프로젝트 팀 「포테토」

---

## 📋 목차

1. 시작 전 주의사항
2. 전체 설치 스택 개요
3. Phase 1 — Windows: NVIDIA 드라이버
4. Phase 2 — WSL2 + Ubuntu 24.04
5. Phase 3 — GPU 스택 (CUDA / cuDNN / TensorRT)
6. Phase 4 — GStreamer + 시스템 라이브러리
7. Phase 5 — DeepStream SDK + pyds
8. Phase 6 — Python 가상환경
9. Phase 7 — LLM 스택 (llama.cpp / Qwen2.5 / LangChain)
10. Phase 8 — Audio 스택 (Whisper / VITS / openWakeWord)
11. Phase 9 — 백엔드 및 알림 (Redis / FastAPI / Firebase / Twilio)
12. 최종 검증 체크리스트
13. 팀 작업 분담 권장
14. 버전 매트릭스 요약

---

## 1. 시작 전 주의사항

### ⚠️ DeepStream의 공식 WSL2 지원은 Docker 방식

이 가이드는 **네이티브 설치**를 전제로 하며, NVIDIA 공식 검증 범위를 벗어납니다. 의존성 충돌 시 팀에서 직접 트러블슈팅해야 함을 이해하고 진행하세요.

### ⚠️ DeepStream 버전 선택

| 버전 | 상태 | 비고 |
|:---:|:---:|---|
| **DeepStream 8.0** | ✅ 권장 | 안정적. Ubuntu 24.04 + CUDA 12.8 + TensorRT 10.9 |
| DeepStream 9.0 | ⚠️ 주의 | 2026-04 현재 TensorRT 10.14 설치 이슈 보고됨. 안정화 대기 권장 |

본 가이드는 **DeepStream 8.0 기준**으로 작성됐습니다.

### ⚠️ GPU 요구사항

- GeForce 또는 Quadro 계열 (Turing 이상 권장, RTX 20/30/40 시리즈)
- Tesla/Datacenter GPU는 WSL2 미지원
- VRAM 4GB 환경에서는 모듈별 분리 개발 필요 (Vision + LLM 동시 실행 불가)

### ⚠️ WSL 내부에 Linux용 NVIDIA 드라이버 절대 설치 금지

Windows 호스트 드라이버가 WSL2 내부에서 `libcuda.so`로 자동 연결됩니다. Linux 드라이버를 설치하면 이 연결을 덮어써서 WSL이 망가집니다.

---

## 2. 전체 설치 스택 개요

| Phase | 계층 | 설치 대상 | 예상 소요 |
|:---:|---|---|:---:|
| 1 | Windows 측 | NVIDIA GPU 드라이버 | 15분 |
| 2 | WSL2 기본 | Ubuntu 24.04, 기본 빌드 도구 | 20분 |
| 3 | GPU 스택 | CUDA Toolkit (WSL용) → cuDNN → TensorRT | 40분 |
| 4 | 멀티미디어 | GStreamer 1.20+ 및 플러그인 | 10분 |
| 5 | DeepStream | DeepStream SDK 본체 + pyds | 30분 |
| 6 | Python 환경 | Python 3.10, venv, 핵심 패키지 | 10분 |
| 7 | LLM 스택 | llama.cpp, Qwen2.5-3B GGUF, LangChain, FAISS, NeMo Guardrails | 30분 |
| 8 | Audio 스택 | Whisper, VITS-KR, openWakeWord, 오디오 라이브러리 | 20분 |
| 9 | 백엔드/알림 | Redis, FastAPI, Firebase Admin, Twilio | 10분 |

**총 소요 시간: 약 3~4시간** (다운로드 속도에 따라 편차 큼)

---

## 3. Phase 1 — Windows: NVIDIA 드라이버

### 3.1 드라이버 다운로드

1. https://www.nvidia.com/Download/index.aspx 접속
2. 본인 GPU 모델 선택 후 **GameReady 드라이버** 다운로드
3. DeepStream 8.0 기준: **드라이버 572.60 이상** 권장

### 3.2 설치 옵션

- 설치 시 **Advanced → Perform Clean Installation** 체크
- 재부팅 후 `PowerShell`에서 다음 명령어 실행:

```powershell
nvidia-smi
```

**체크포인트**: GPU 이름, VRAM, 드라이버 버전이 표시되면 성공.

---

## 4. Phase 2 — WSL2 + Ubuntu 24.04

### 4.1 WSL 설치 (PowerShell 관리자 권한)

```powershell
wsl --install Ubuntu-24.04
wsl --list --verbose
```

VERSION 컬럼이 **2**로 표시되어야 합니다.

### 4.2 Ubuntu 기본 세팅 (WSL 터미널)

```bash
sudo apt update && sudo apt upgrade -y

# 빌드 도구 + 기본 라이브러리
sudo apt install -y build-essential cmake git wget curl \
    pkg-config software-properties-common ca-certificates \
    lsb-release gnupg2 apt-transport-https
```

**체크포인트**: `gcc --version`, `cmake --version` 실행 시 버전 정보 표시.

---

## 5. Phase 3 — GPU 스택

### 5.1 CUDA Toolkit 12.6 (WSL2 전용 저장소 사용)

⚠️ **반드시 wsl-ubuntu 저장소**를 사용해야 합니다. 일반 Linux용 저장소를 쓰면 NVIDIA 드라이버까지 설치되어 WSL이 망가집니다.

```bash
# CUDA WSL2 전용 저장소 추가
wget https://developer.download.nvidia.com/compute/cuda/repos/wsl-ubuntu/x86_64/cuda-keyring_1.1-1_all.deb
sudo dpkg -i cuda-keyring_1.1-1_all.deb
sudo apt update

# CUDA Toolkit만 설치 (드라이버 제외)
sudo apt install -y cuda-toolkit-12-6

# 환경 변수 설정
echo 'export PATH=/usr/local/cuda/bin:$PATH' >> ~/.bashrc
echo 'export LD_LIBRARY_PATH=/usr/local/cuda/lib64:$LD_LIBRARY_PATH' >> ~/.bashrc
source ~/.bashrc
```

**검증**:

```bash
nvcc --version
nvidia-smi
```

### 5.2 cuDNN 9.8

1. https://developer.nvidia.com/cudnn 에서 **Ubuntu 24.04용 cuDNN 9.8** 다운로드 (NVIDIA Developer 계정 필요)
2. 다운로드한 `.deb` 파일을 WSL로 복사

```bash
# 파일명은 실제 다운로드한 이름에 맞춰 수정
sudo dpkg -i cudnn-local-repo-ubuntu2404-9.8.0_1.0-1_amd64.deb
sudo cp /var/cudnn-local-repo-*/cudnn-*-keyring.gpg /usr/share/keyrings/
sudo apt update
sudo apt install -y cudnn
```

### 5.3 TensorRT 10.9

1. https://developer.nvidia.com/tensorrt 에서 **Ubuntu 24.04 + CUDA 12.8용 TensorRT 10.9** 다운로드
2. 파일을 WSL로 복사 후 설치

```bash
sudo dpkg -i nv-tensorrt-local-repo-ubuntu2404-10.9.0-cuda-12.8_1.0-1_amd64.deb
sudo cp /var/nv-tensorrt-local-repo-*/nv-tensorrt-*-keyring.gpg /usr/share/keyrings/
sudo apt update
sudo apt install -y tensorrt python3-libnvinfer-dev
```

**검증**:

```bash
dpkg -l | grep TensorRT
# libnvinfer10, tensorrt 관련 패키지 목록이 출력되어야 함
```

---

## 6. Phase 4 — GStreamer + 시스템 라이브러리

```bash
sudo apt install -y \
    libssl3 libssl-dev libgles2-mesa-dev \
    libgstreamer1.0-0 gstreamer1.0-tools \
    gstreamer1.0-plugins-good gstreamer1.0-plugins-bad \
    gstreamer1.0-plugins-ugly gstreamer1.0-libav \
    libgstreamer-plugins-base1.0-dev libgstrtspserver-1.0-0 \
    libgstreamer1.0-dev libgirepository1.0-dev \
    libjansson4 libyaml-cpp-dev libjsoncpp-dev \
    protobuf-compiler python3-gi python3-dev \
    python3-gst-1.0 python-gi-dev libglib2.0-dev libcairo2-dev
```

**검증**:

```bash
gst-launch-1.0 --version
# GStreamer 1.24.x 또는 1.20+ 표시되어야 함
```

---

## 7. Phase 5 — DeepStream SDK 설치

### 7.1 DeepStream SDK 본체

1. https://developer.nvidia.com/deepstream-sdk 에서 **DeepStream 8.0 dGPU Debian package** 다운로드
   - 파일명 예시: `deepstream-8.0_8.0.0-1_amd64.deb`
2. WSL로 복사 후 설치

```bash
# 이전 DeepStream 제거 (처음 설치면 스킵)
sudo rm -rf /usr/local/deepstream /opt/nvidia/deepstream/deepstream*

# 설치
sudo apt install -y ./deepstream-8.0_8.0.0-1_amd64.deb
sudo ldconfig

# 추가 의존성 설치 (필수)
cd /opt/nvidia/deepstream/deepstream/
sudo ./user_additional_install.sh
```

**검증**:

```bash
deepstream-app --version-all
gst-inspect-1.0 nvinfer
# nvinfer 플러그인 정보가 출력되면 성공
```

### 7.2 pyds (Python Bindings)

```bash
cd /opt/nvidia/deepstream/deepstream/sources/deepstream_python_apps/bindings

# Wheel이 미리 빌드되어 있는 경우
pip3 install ./pyds-*.whl

# Wheel이 없으면 직접 빌드 (README 참조)
```

---

## 8. Phase 6 — Python 가상환경

Ubuntu 24.04 기본 Python은 3.12이지만, 본 프로젝트는 **Python 3.10**을 사용합니다 (계획서 기준).

```bash
# Python 3.10 설치
sudo add-apt-repository ppa:deadsnakes/ppa -y
sudo apt update
sudo apt install -y python3.10 python3.10-venv python3.10-dev

# 프로젝트 디렉터리 생성
mkdir -p ~/mind-care-vision && cd ~/mind-care-vision

# 가상환경 생성
python3.10 -m venv .venv
source .venv/bin/activate

# 기본 패키지
pip install --upgrade pip setuptools wheel
pip install numpy opencv-python pillow pyyaml tqdm
```

**주의**: 이후 모든 `pip install` 명령은 `.venv` 활성화 상태에서 실행해야 합니다.

```bash
# 항상 작업 시작 전에 실행
cd ~/mind-care-vision
source .venv/bin/activate
```

---

## 9. Phase 7 — LLM 스택

### 9.1 llama.cpp (Qwen2.5-3B GGUF 추론 엔진)

```bash
cd ~
git clone https://github.com/ggerganov/llama.cpp
cd llama.cpp

# CUDA 지원 빌드 (4GB VRAM에서 일부 레이어 GPU offload 가능)
cmake -B build -DGGML_CUDA=ON
cmake --build build --config Release -j$(nproc)
```

### 9.2 Qwen2.5-3B-Instruct 모델 다운로드

```bash
cd ~/llama.cpp
mkdir -p models && cd models

# Q4_K_M 양자화 버전 (약 2GB, 4GB VRAM에 적합)
wget https://huggingface.co/Qwen/Qwen2.5-3B-Instruct-GGUF/resolve/main/qwen2.5-3b-instruct-q4_k_m.gguf
```

### 9.3 llama.cpp 서버 실행 테스트

```bash
cd ~/llama.cpp
./build/bin/llama-server \
    -m models/qwen2.5-3b-instruct-q4_k_m.gguf \
    --n-gpu-layers 20 \
    --port 8080
```

브라우저에서 `http://localhost:8080` 접속해 채팅이 되면 성공.

### 9.4 LangChain + RAG 스택

```bash
# venv 활성화 상태 확인 후
source ~/mind-care-vision/.venv/bin/activate

pip install langchain langchain-community langchain-openai
pip install faiss-cpu
pip install sentence-transformers
pip install nemoguardrails
pip install pypdf markdown
```

**참고**: FAISS는 GPU 버전(`faiss-gpu`)도 있지만 CUDA 버전 매칭이 까다로워 CPU 버전 권장. 벡터 수가 100만 미만이면 CPU로 충분.

---

## 10. Phase 8 — Audio 스택

### 10.1 오디오 시스템 라이브러리

```bash
sudo apt install -y portaudio19-dev libsndfile1 ffmpeg alsa-utils
```

### 10.2 Python 오디오 패키지

```bash
source ~/mind-care-vision/.venv/bin/activate

# 기본 오디오 I/O
pip install sounddevice soundfile librosa pydub

# STT: Whisper
pip install openai-whisper

# TTS: Coqui TTS (VITS 포함)
pip install TTS

# Wake word
pip install openwakeword
pip install webrtcvad
```

### 10.3 WSL2 오디오 주의사항

> ⚠️ **WSL2는 기본적으로 마이크·스피커 접근이 제한됩니다.**
> - **개발 PC에서는 녹음된 `.wav` 파일로 테스트**
> - 실시간 오디오 I/O는 **Jetson AGX Xavier 배포 시** 검증
> - 또는 WSL2에 PulseAudio를 연결하는 고급 설정 사용 (복잡함)

---

## 11. Phase 9 — 백엔드 및 알림

### 11.1 Redis (이벤트 버스)

```bash
sudo apt install -y redis-server
sudo service redis-server start

# 검증
redis-cli ping
# "PONG" 응답 시 성공
```

**자동 시작 설정**:

```bash
# WSL 부팅 시 자동 실행되도록 ~/.bashrc에 추가
echo 'sudo service redis-server start > /dev/null 2>&1' >> ~/.bashrc
```

### 11.2 FastAPI + 백엔드 라이브러리

```bash
source ~/mind-care-vision/.venv/bin/activate

pip install fastapi uvicorn[standard]
pip install redis
pip install sqlalchemy
pip install python-multipart
pip install httpx aiofiles
```

### 11.3 알림 서비스 SDK

```bash
pip install firebase-admin   # FCM 푸시
pip install twilio           # SMS
```

**외부 계정 필요**:
- Firebase: https://console.firebase.google.com → 프로젝트 생성 → 서비스 계정 JSON 키 발급
- Twilio: https://www.twilio.com → 계정 생성 → API Key + 발신 번호 발급

---

## 12. 최종 검증 체크리스트

모든 Phase 완료 후 아래 명령을 순서대로 실행하여 검증하세요.

```bash
# ① GPU 드라이버 및 CUDA
nvidia-smi
nvcc --version

# ② cuDNN 및 TensorRT
dpkg -l | grep -E "cudnn|TensorRT"

# ③ GStreamer
gst-launch-1.0 --version

# ④ DeepStream
deepstream-app --version-all
gst-inspect-1.0 nvinfer

# ⑤ Python 환경
source ~/mind-care-vision/.venv/bin/activate
python --version   # Python 3.10.x
pip list | grep -E "langchain|faiss|whisper|fastapi|redis"

# ⑥ Redis
redis-cli ping

# ⑦ llama.cpp
~/llama.cpp/build/bin/llama-cli --version
```

### DeepStream 샘플 실행 테스트

```bash
cd /opt/nvidia/deepstream/deepstream/samples/streams

# 샘플 영상 디코딩 → 재인코딩 → 파일 저장
# (WSL2에서는 화면 출력이 불안정하므로 filesink 사용)
gst-launch-1.0 filesrc location=sample_720p.mp4 ! qtdemux ! h264parse \
    ! nvv4l2decoder ! nvv4l2h264enc ! h264parse ! qtmux \
    ! filesink location=/tmp/filedump.mp4 -v
```

`/tmp/filedump.mp4` 생성 성공 시 **DeepStream 전체 설치 완료**.

Windows 탐색기에서 `\\wsl$\Ubuntu-24.04\tmp\filedump.mp4` 로 접근해 재생 가능 여부 확인.

---

## 13. 팀 작업 분담 권장

프로젝트 계획서 상 역할 기준으로 Phase 분담:

| 담당자 | 역할 | 담당 Phase |
|---|---|---|
| PM | PM, 데이터 엔지니어링, 계획서 작성 | Phase 1~2 (환경), Phase 9 (백엔드) |
| 파이프라인 구축 담당 | 파이프라인 구축 | Phase 3~5 (GPU + DeepStream) |
| 영상 처리 모델 담당 | 영상 처리 모델 구축 | Phase 5 이후 YOLOv8/SCRFD 배포 |
| 언어 모델 담당 | 언어 모델 구축 | Phase 7 (llama.cpp + Qwen) |

**핵심 원칙**:
- Phase 3 (GPU 스택)은 **한 사람이 끝까지 책임지고 검증**. CUDA/cuDNN/TensorRT 버전 매칭 실패가 가장 흔한 이슈.
- 각 팀원은 자기 Phase 완료 후 **다른 팀원의 환경에서도 작동하는지 크로스 체크** 필수.

---

## 14. 버전 매트릭스 요약

팀원 간 환경을 일치시키기 위한 버전 기준표.

| 컴포넌트 | 버전 | 비고 |
|---|---|---|
| Windows | 10 21H2+ 또는 11 | |
| NVIDIA Driver (Windows) | ≥ 572.60 | GameReady 권장 |
| WSL | 2 | `wsl --list -v`로 확인 |
| Ubuntu | 24.04 LTS | DeepStream 8.0 공식 지원 |
| Python | 3.10.x | `deadsnakes` PPA |
| CUDA Toolkit | 12.6 | **wsl-ubuntu 저장소 전용** |
| cuDNN | 9.8 | |
| TensorRT | 10.9 | cuda-12.8 빌드 |
| GStreamer | 1.24 (또는 1.20+) | Ubuntu 기본 패키지 |
| DeepStream SDK | 8.0 | 9.0은 안정화 대기 |
| Redis | 기본 (apt) | |

---

## 📎 부록: 자주 발생하는 문제

### ① `nvidia-smi`는 되는데 `nvcc`가 "command not found"

PATH가 적용되지 않음. 다음 실행:
```bash
source ~/.bashrc
# 또는 새 WSL 터미널 재접속
```

### ② `deepstream-app`이 "Failed to load plugin" 에러

```bash
# GStreamer 캐시 재생성
rm -rf ~/.cache/gstreamer-1.0
sudo ldconfig
```

### ③ pip 설치 시 "externally-managed-environment" 에러

venv를 활성화하지 않은 상태. 다음 실행:
```bash
source ~/mind-care-vision/.venv/bin/activate
```

### ④ WSL에서 Windows 파일 접근이 느림

**Windows 경로(`/mnt/c/...`)에 프로젝트 저장 금지**. 반드시 WSL 내부 경로(`~/` 이하)에 저장하세요. 성능 차이 10배 이상.

### ⑤ WSL 디스크 용량 부족

```powershell
# PowerShell에서 WSL 종료
wsl --shutdown

# 디스크 압축 (VHD 파일 위치는 환경마다 다름)
# Settings → System → Storage → WSL 앱에서 확인 가능
```

---

## 📚 참고 링크

- NVIDIA DeepStream SDK: https://developer.nvidia.com/deepstream-sdk
- DeepStream 공식 문서: https://docs.nvidia.com/metropolis/deepstream/dev-guide/
- CUDA on WSL 가이드: https://docs.nvidia.com/cuda/wsl-user-guide/
- Qwen2.5 GGUF 모델: https://huggingface.co/Qwen/Qwen2.5-3B-Instruct-GGUF
- llama.cpp: https://github.com/ggerganov/llama.cpp
- openWakeWord: https://github.com/dscripka/openWakeWord
- Coqui TTS: https://github.com/coqui-ai/TTS

---

**문서 버전**: v1.0
**마지막 업데이트**: 2026-04-20
